# Analysis 1 — Gap=1 → TFU (EDA)

**Population (month M-1):** `is_donated = 1 OR is_follow_bet = 1`  
**Target (month M):** `is_tfu = 1`

Source table: `tfu_user_monthly` (one row per `cust_id × data_month`).

Sections:
1. Population overview
2. Segment distributions (Donated vs Follow Bet, side by side)
3. Behavioral profile — 4-way: Donated×{conv, non-conv}, Follow Bet×{conv, non-conv}
4. Correlation heatmaps (one per sub-segment)
5. Monthly trends + cohort stickiness

## 0. Setup

In [ ]:
# --- Colab auth + BigQuery client ---
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')

# TODO: fill in your project / dataset
PROJECT_ID = 'nf-bifrost'
DATASET    = 'your_dataset'                       # <-- update
TABLE      = f'{PROJECT_ID}.{DATASET}.tfu_user_monthly'

client = bigquery.Client(project=PROJECT_ID)

## 1. Pull `tfu_user_monthly` and derive flags

Schema columns (from BQ build):
- **IDs/time:** `cust_id`, `data_month`, `month_year`, `month_end`
- **Account:** `account_age_days`, `account_age_tier`, `site`, `currency`
- **Loyalty:** `sessions_count`, `distinct_streamers`, `sessions_bucket`
- **Watch:** `total_watch_sec`, `avg_watch_sec_per_session`, `watch_bucket`
- **Chat:** `total_messages`, `chat_sessions`, `total_bullet_sec`, `total_chatroom_sec`
- **Gifting:** `total_tip_count/usd`, `total_box_count/usd`, `total_wheel_count/usd`
- **Betting:** `total_bet_count`, `total_member_to`, `total_bdw_bet_count`, `total_follow_bet_count`
- **Preferences:** `stream_type_pref`, `device_pref`, `time_segment`, `day_segment`, `breadth_score`
- **Streamer affinity:** `top_follow_streamer*`, `top_gift_streamer*`, `top_streamer_is_same`
- **Target:** `is_tfu`

We derive `is_donated`, `is_follow_bet`, `tfu_gap`, and `sub_segment` in pandas.

In [ ]:
sql = f"""
SELECT *
FROM `{TABLE}`
"""
df = client.query(sql).to_dataframe()
df['data_month'] = pd.to_datetime(df['data_month'])
print('rows:', len(df), ' users:', df.cust_id.nunique(), ' months:', sorted(df.data_month.unique()))

In [ ]:
# --- Derive sub-segment flags ---
gift_any  = (df['total_tip_count'].fillna(0) + df['total_box_count'].fillna(0) + df['total_wheel_count'].fillna(0)) > 0
follow_bet_any = df['total_follow_bet_count'].fillna(0) > 0

df['is_gift']       = gift_any.astype(int)
df['is_follow_bet'] = (follow_bet_any & ~gift_any).astype(int)
df['is_donated']    = (gift_any & ~follow_bet_any).astype(int)
# sanity: is_tfu should equal (gift_any & follow_bet_any)
df['is_tfu_check']  = (gift_any & follow_bet_any).astype(int)

# tfu_gap: 0=TFU, 1=Donated or Follow Bet, 2=Cold
df['tfu_gap'] = np.where(df['is_tfu'] == 1, 0,
                  np.where((df['is_donated'] == 1) | (df['is_follow_bet'] == 1), 1, 2))

df['sub_segment'] = np.select(
    [df['is_tfu'] == 1, df['is_donated'] == 1, df['is_follow_bet'] == 1],
    ['TFU', 'Donated', 'Follow Bet'],
    default='Cold'
)

# Sanity check
mismatch = (df['is_tfu'] != df['is_tfu_check']).sum()
print(f'is_tfu sanity mismatches (should be 0): {mismatch}')
df.groupby('data_month')['sub_segment'].value_counts().unstack(fill_value=0)

## 2. Attach next-month TFU status to each gap=1 user-month

Population: rows where `is_donated = 1 OR is_follow_bet = 1` in month M.  
For each such row, look up whether the same `cust_id` was TFU in month M+1.  
Users absent in M+1 are treated as `is_tfu_next = 0`. The final month is dropped (no M+1 to look up).

In [ ]:
df = df.sort_values(['cust_id', 'data_month'])

# For each gap=1 user-month, look up is_tfu in the following month
gap1 = df[(df['is_donated'] == 1) | (df['is_follow_bet'] == 1)].copy()
gap1['next_month'] = gap1['data_month'] + pd.offsets.MonthBegin(1)

nxt = df[['cust_id', 'data_month', 'is_tfu']].rename(
    columns={'data_month': 'next_month', 'is_tfu': 'is_tfu_next'}
)

panel = gap1.merge(nxt, on=['cust_id', 'next_month'], how='left')
panel['is_tfu_next'] = panel['is_tfu_next'].fillna(0).astype(int)

# Drop the last month (no next month available to score)
max_month = df['data_month'].max()
panel = panel[panel['data_month'] < max_month].copy()

print('Gap=1 panel rows:', len(panel))
print('Months covered :', sorted(panel.data_month.dt.strftime('%Y-%m').unique()))
print('Overall conv. rate to TFU:', panel['is_tfu_next'].mean().round(4))

## Section 1 — Population overview

Donated vs Follow Bet count per month, conversion rate to TFU, trend over 6 months.

In [ ]:
# Counts of Donated vs Follow Bet per month + conversion rate
pop = (panel.groupby(['data_month', 'sub_segment'])
             .agg(users=('cust_id', 'nunique'),
                  converters=('is_tfu_next', 'sum'))
             .reset_index())
pop['conv_rate'] = pop['converters'] / pop['users']
pop

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=pop, x='data_month', y='users', hue='sub_segment', ax=axes[0])
axes[0].set_title('Gap=1 population size per month')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('users')

sns.lineplot(data=pop, x='data_month', y='conv_rate', hue='sub_segment',
             marker='o', ax=axes[1])
axes[1].set_title('Conversion rate to TFU (next month)')
axes[1].set_ylabel('conv_rate')
axes[1].tick_params(axis='x', rotation=45)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))

plt.tight_layout(); plt.show()

In [ ]:
# Headline: pooled conversion rate per sub-segment
summary = (panel.groupby('sub_segment')
                 .agg(users=('cust_id', 'count'),
                      conv_rate=('is_tfu_next', 'mean'))
                 .round(4))
summary

## Section 2 — Segment distributions (Donated vs Follow Bet, side-by-side)

Compare the two sub-segments across: account age tier, watch bucket, time/day segments, stream type, device, sessions bucket.

In [ ]:
categorical_cols = [
    'account_age_tier', 'watch_bucket', 'sessions_bucket',
    'time_segment', 'day_segment',
    'stream_type_pref', 'device_pref',
]

def share_table(panel, col):
    t = (panel.groupby(['sub_segment', col]).size()
                .groupby(level=0).apply(lambda s: s / s.sum())
                .rename('share').reset_index())
    return t

for col in categorical_cols:
    t = share_table(panel, col)
    plt.figure(figsize=(8, 3.5))
    sns.barplot(data=t, x=col, y='share', hue='sub_segment')
    plt.title(f'Distribution of {col} — Donated vs Follow Bet')
    plt.ylabel('share of sub-segment')
    plt.xticks(rotation=30, ha='right')
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    plt.tight_layout(); plt.show()

In [ ]:
# Conversion rate by segment, split by sub-segment
for col in categorical_cols:
    t = (panel.groupby(['sub_segment', col])['is_tfu_next']
               .mean().rename('conv_rate').reset_index())
    plt.figure(figsize=(8, 3.5))
    sns.barplot(data=t, x=col, y='conv_rate', hue='sub_segment')
    plt.title(f'TFU conversion rate by {col}')
    plt.xticks(rotation=30, ha='right')
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
    plt.tight_layout(); plt.show()

## Section 3 — Behavioral profile (4-way comparison)

Cohorts:
- Donated → converters (became TFU)
- Donated → non-converters
- Follow Bet → converters
- Follow Bet → non-converters

In [ ]:
panel['cohort'] = (panel['sub_segment'] + np.where(panel['is_tfu_next'] == 1,
                                                    ' → TFU',
                                                    ' → no'))
panel['cohort'].value_counts()

In [ ]:
numeric_features = [
    # Watch
    'total_watch_sec', 'avg_watch_sec_per_session',
    # Chat
    'total_messages', 'chat_sessions', 'total_bullet_sec', 'total_chatroom_sec',
    # Gifting
    'total_tip_count', 'total_box_count', 'total_wheel_count',
    'total_tip_usd', 'total_box_usd', 'total_wheel_usd',
    # Betting
    'total_bet_count', 'total_member_to', 'total_bdw_bet_count', 'total_follow_bet_count',
    # Breadth / loyalty
    'breadth_score', 'sessions_count', 'distinct_streamers',
]
# Add a convenience: total gift USD
panel['total_gift_usd'] = (panel['total_tip_usd'].fillna(0)
                           + panel['total_box_usd'].fillna(0)
                           + panel['total_wheel_usd'].fillna(0))
numeric_features.append('total_gift_usd')

In [ ]:
# Summary table: median + mean per cohort
profile = (panel.groupby('cohort')[numeric_features]
                 .agg(['median', 'mean'])
                 .round(2))
profile

In [ ]:
# Boxplots (log-scale) per feature, 4-way cohort
cohort_order = ['Donated → TFU', 'Donated → no', 'Follow Bet → TFU', 'Follow Bet → no']

n = len(numeric_features)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    ax = axes[i]
    data = panel[[feat, 'cohort']].copy()
    data[feat] = data[feat].clip(lower=0) + 1  # log-friendly
    sns.boxplot(data=data, x='cohort', y=feat, order=cohort_order,
                ax=ax, showfliers=False)
    ax.set_yscale('log')
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout(); plt.show()

## Section 4 — Correlation heatmaps (per sub-segment)

Numeric features + target `is_tfu_next` — one heatmap for Donated, one for Follow Bet.

In [ ]:
def corr_heatmap(panel, sub_segment_name):
    sub = panel[panel['sub_segment'] == sub_segment_name].copy()
    cols = numeric_features + ['is_tfu_next']
    corr = sub[cols].corr(method='spearman')
    plt.figure(figsize=(11, 9))
    sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
                annot=False, square=False)
    plt.title(f'Spearman correlation — {sub_segment_name} (n={len(sub)})')
    plt.tight_layout(); plt.show()

    # Top correlations with target
    top = (corr['is_tfu_next'].drop('is_tfu_next')
                              .abs().sort_values(ascending=False)
                              .head(15))
    print(f'\nTop |corr| with is_tfu_next — {sub_segment_name}:')
    print(top.round(3))
    return corr

_ = corr_heatmap(panel, 'Donated')
_ = corr_heatmap(panel, 'Follow Bet')

## Section 5 — Monthly trends + cohort stickiness

- TFU conversion rate by sub-segment over the 6 months.
- Cohort stickiness: of users in Donated/Follow Bet in month M, how many remain in the same state in M+1 (vs convert / drop out).

In [ ]:
# Conversion rate trend already computed in Section 1 — plot a cleaner version
plt.figure(figsize=(9, 4))
sns.lineplot(data=pop, x='data_month', y='conv_rate', hue='sub_segment',
             marker='o', linewidth=2)
plt.title('TFU conversion rate by sub-segment — 6-month trend')
plt.ylabel('conv_rate')
plt.xticks(rotation=45)
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
plt.tight_layout(); plt.show()

In [ ]:
# Cohort stickiness: state in M vs state in M+1
state_now  = df[['cust_id', 'data_month', 'sub_segment']].copy()
state_now['next_month'] = state_now['data_month'] + pd.offsets.MonthBegin(1)

state_next = df[['cust_id', 'data_month', 'sub_segment']].rename(
    columns={'data_month': 'next_month', 'sub_segment': 'sub_segment_next'}
)

trans = state_now.merge(state_next, on=['cust_id', 'next_month'], how='left')
trans['sub_segment_next'] = trans['sub_segment_next'].fillna('Inactive')
trans = trans[trans['sub_segment'].isin(['Donated', 'Follow Bet'])]

transition = (trans.groupby(['sub_segment', 'sub_segment_next']).size()
                    .groupby(level=0).apply(lambda s: s / s.sum())
                    .rename('share').reset_index())
transition_pivot = transition.pivot(index='sub_segment',
                                     columns='sub_segment_next',
                                     values='share').fillna(0).round(3)
transition_pivot

In [ ]:
plt.figure(figsize=(7, 3))
sns.heatmap(transition_pivot, annot=True, fmt='.1%', cmap='Blues',
            cbar_kws={'format': plt.FuncFormatter(lambda y, _: f'{y:.0%}')})
plt.title('Sub-segment transitions: month M → M+1')
plt.ylabel('state in M')
plt.xlabel('state in M+1')
plt.tight_layout(); plt.show()

## Takeaways scratch-pad

Fill in after running:
- Donated conv rate to TFU ≈ _
- Follow Bet conv rate to TFU ≈ _
- Strongest correlated features with TFU next month — Donated: _
- Strongest correlated features with TFU next month — Follow Bet: _
- Stickiness: % of Donated stays Donated next month _, drops to Cold/Inactive _
- Stickiness: % of Follow Bet stays Follow Bet next month _, drops to Cold/Inactive _
- Highest-conversion segment(s) by account age tier / watch bucket / device: _